## PML Book - Chapter 4 demo

#### adapted from J. MacAuley's notebook 

A walk-through of the original notebook can be found here: https://cseweb.ucsd.edu/classes/fa19/cse258-a/pdp/implementing_recommender.pdf

In [97]:
import os
import gzip
import math
import random
from collections import defaultdict


#### Input Dataset

Amazon musical instrument review data. Originally from https://s3.amazonaws.com/amazon-reviews-pds/tsv/index.txt 

Data is available at http://cseweb.ucsd.edu/~jmcauley/pml/data/. 
- Download and save to your own directory.


In [98]:
dataDir = './'
path = os.path.join(
    dataDir, "amazon_reviews_us_Musical_Instruments_v1_00.tsv.gz")
f = gzip.open(path, 'rt', encoding="utf8")

header = f.readline()
header = header.strip().split('\t')


Dataset contains the following fields

In [99]:
header


['marketplace',
 'customer_id',
 'review_id',
 'product_id',
 'product_parent',
 'product_title',
 'product_category',
 'star_rating',
 'helpful_votes',
 'total_votes',
 'vine',
 'verified_purchase',
 'review_headline',
 'review_body',
 'review_date']

Parse the data and convert fields to integers where needed

In [100]:
dataset = []

for line in f:
    fields = line.strip().split('\t')
    d = dict(zip(header, fields))
    d['star_rating'] = int(d['star_rating'])
    d['helpful_votes'] = int(d['helpful_votes'])
    d['total_votes'] = int(d['total_votes'])
    dataset.append(d)


One row of the dataset (as a python dictionary)

In [101]:
dataset[0]


{'marketplace': 'US',
 'customer_id': '45610553',
 'review_id': 'RMDCHWD0Y5OZ9',
 'product_id': 'B00HH62VB6',
 'product_parent': '618218723',
 'product_title': 'AGPtek® 10 Isolated Output 9V 12V 18V Guitar Pedal Board Power Supply Effect Pedals with Isolated Short Cricuit / Overcurrent Protection',
 'product_category': 'Musical Instruments',
 'star_rating': 3,
 'helpful_votes': 0,
 'total_votes': 1,
 'vine': 'N',
 'verified_purchase': 'N',
 'review_headline': 'Three Stars',
 'review_body': 'Works very good, but induces ALOT of noise.',
 'review_date': '2015-08-31'}

Extract a few utility data structures

In [102]:
usersPerItem = defaultdict(set)  # Maps an item to the users who rated it (item profile)
itemsPerUser = defaultdict(set)  # Maps a user to the items that they rated (user profile)

itemNames = {}
ratingDict = {}  # To retrieve a rating for a specific user/item pair

for d in dataset:
    user, item = d['customer_id'], d['product_id']
    usersPerItem[item].add(user)
    itemsPerUser[user].add(item)
    ratingDict[(user, item)] = d['star_rating']
    itemNames[item] = d['product_title']


Extract per-user and per-item averages (useful later for rating prediction)

In [103]:
userAverages = {}
itemAverages = {}

for u in itemsPerUser:
    rs = [ratingDict[(u, i)] for i in itemsPerUser[u]]
    userAverages[u] = sum(rs) / len(rs)

for i in usersPerItem:
    rs = [ratingDict[(u, i)] for u in usersPerItem[i]]
    itemAverages[i] = sum(rs) / len(rs)


### Similarity metrics

#### Jaccard

In [104]:
def Jaccard(s1, s2):
    numer = len(s1.intersection(s2))
    denom = len(s1.union(s2))
    if denom == 0:
        return 0
    return numer / denom


#### Cosine

Simple implementation for set-structured data

In [105]:
def CosineSet(s1, s2):
    # Not a proper implementation, operates on sets so correct for interactions only
    numer = len(s1.intersection(s2))
    denom = math.sqrt(len(s1)) * math.sqrt(len(s2))
    if denom == 0:
        return 0
    return numer / denom


Or for real values (e.g. ratings). Note that this implementation uses global variables (usersPerItem, ratingDict), which ideally should be passed as parameters.

In [106]:
def Cosine(i1, i2):
    # Between two items
    inter = usersPerItem[i1].intersection(usersPerItem[i2])
    numer = 0
    denom1 = 0
    denom2 = 0
    for u in inter:
        numer += ratingDict[(u, i1)]*ratingDict[(u, i2)]
    for u in usersPerItem[i1]:
        denom1 += ratingDict[(u, i1)]**2
    for u in usersPerItem[i2]:
        denom2 += ratingDict[(u, i2)]**2
    denom = math.sqrt(denom1) * math.sqrt(denom2)
    if denom == 0:
        return 0
    return numer / denom


#### Pearson

In [107]:
def Pearson(i1, i2):
    # Between two items
    iBar1 = itemAverages[i1]
    iBar2 = itemAverages[i2]
    inter = usersPerItem[i1].intersection(usersPerItem[i2])
    numer = 0
    denom1 = 0
    denom2 = 0
    for u in inter:
        numer += (ratingDict[(u, i1)] - iBar1)*(ratingDict[(u, i2)] - iBar2)
    for u in inter:  # usersPerItem[i1]:
        denom1 += (ratingDict[(u, i1)] - iBar1)**2
    # for u in usersPerItem[i2]:
        denom2 += (ratingDict[(u, i2)] - iBar2)**2
    denom = math.sqrt(denom1) * math.sqrt(denom2)
    if denom == 0:
        return 0
    return numer / denom


## Retrieve the most similar items to a given query

In [108]:
#retrieve N items most similar to a candidate item i

def mostSimilar(i, N):   
    similarities = []
    users = usersPerItem[i]  #find all users who purchased i
    for i2 in usersPerItem:  #iterate  over all other items (profiles) and compute their similarity to i in terms of common users
        if i2 == i:
            continue
        sim = Jaccard(users, usersPerItem[i2])  
        #sim = Pearson(i, i2) # Could use alternate similarity metrics straightforwardly
        similarities.append((sim, i2))
    similarities.sort(reverse=True)  
    return similarities[:N]


Choose an item to use as a query

In [109]:
dataset[2]


{'marketplace': 'US',
 'customer_id': '6111003',
 'review_id': 'RIZR67JKUDBI0',
 'product_id': 'B0006VMBHI',
 'product_parent': '603261968',
 'product_title': 'AudioQuest LP record clean brush',
 'product_category': 'Musical Instruments',
 'star_rating': 3,
 'helpful_votes': 0,
 'total_votes': 1,
 'vine': 'N',
 'verified_purchase': 'Y',
 'review_headline': 'Three Stars',
 'review_body': 'removes dust. does not clean',
 'review_date': '2015-08-31'}

In [110]:
query = dataset[2]['product_id']


Retrieve the most similary items

In [111]:
ms = mostSimilar(query, 10)


In [112]:
ms

#prints top-10 list: (sim, product_id)

[(0.028446389496717725, 'B00006I5SD'),
 (0.01694915254237288, 'B00006I5SB'),
 (0.015065913370998116, 'B000AJR482'),
 (0.014204545454545454, 'B00E7MVP3S'),
 (0.008955223880597015, 'B001255YL2'),
 (0.008849557522123894, 'B003EIRVO8'),
 (0.008333333333333333, 'B0015VEZ22'),
 (0.00821917808219178, 'B00006I5UH'),
 (0.008021390374331552, 'B00008BWM7'),
 (0.007656967840735069, 'B000H2BC4E')]

Print names of query and recommended items

In [113]:
print("Input query: ", itemNames[query])
print("")
print("Recommended items: ")
[itemNames[x[1]] for x in ms]


Input query:  AudioQuest LP record clean brush

Recommended items: 


['Shure SFG-2 Stylus Tracking Force Gauge',
 'Shure M97xE High-Performance Magnetic Phono Cartridge',
 'ART Pro Audio DJPRE II Phono Turntable Preamplifier',
 'Signstek Blue LCD Backlight Digital Long-Playing LP Turntable Stylus Force Scale Gauge Tester',
 'Audio Technica AT120E/T Standard Mount Phono Cartridge',
 'Technics: 45 Adaptor for Technics 1200 (SFWE010)',
 'GruvGlide GRUVGLIDE DJ Package',
 'STANTON MAGNETICS Record Cleaner Kit',
 'Shure M97xE High-Performance Magnetic Phono Cartridge',
 'Behringer PP400 Ultra Compact Phono Preamplifier']

##### Question

The previous implementation was not very fast. Which is the slowest component?


The slowest part of the implementation would likely be the iteration over all users with ratings of the current item. The jacardd similarity function is already an O(n) operation by finding the intersecton and length of the two sets. Afterwards, the similarity function would then interate over all users who rated the same item which worst case would be another O(n) bringing the overall time complexity to O(n^2). By going over all users who rated both items, we can skip the ones who would not otherwise have similarity

### Faster implementation

In [114]:
def mostSimilarFast(i, N):
    similarities = []
    users = usersPerItem[i] 
    candidateItems = set()
    for u in users:
        candidateItems = candidateItems.union(itemsPerUser[u]) #iterate over items that were purchased by users who also purchased i 
    for i2 in candidateItems:
        if i2 == i:
            continue
        sim = Jaccard(users, usersPerItem[i2])
        similarities.append((sim, i2))
    similarities.sort(reverse=True)
    return similarities[:N]


Confirm that results are the same...

In [115]:
mostSimilarFast(query, 10)


[(0.028446389496717725, 'B00006I5SD'),
 (0.01694915254237288, 'B00006I5SB'),
 (0.015065913370998116, 'B000AJR482'),
 (0.014204545454545454, 'B00E7MVP3S'),
 (0.008955223880597015, 'B001255YL2'),
 (0.008849557522123894, 'B003EIRVO8'),
 (0.008333333333333333, 'B0015VEZ22'),
 (0.00821917808219178, 'B00006I5UH'),
 (0.008021390374331552, 'B00008BWM7'),
 (0.007656967840735069, 'B000H2BC4E')]

## Similarity-based rating estimation

Use our similarity functions to estimate ratings. Start by building a few utility data structures.

In [116]:
reviewsPerUser = defaultdict(list)
reviewsPerItem = defaultdict(list)


In [117]:
for d in dataset:
    user, item = d['customer_id'], d['product_id']
    reviewsPerUser[user].append(d)
    reviewsPerItem[item].append(d)


In [118]:
ratingMean = sum([d['star_rating'] for d in dataset]) / len(dataset)


In [119]:
ratingMean  #avg rating of the entire dataset


4.251102772543146

Rating prediction heuristic (several alternatives from Chapter 4 could be used)

In [120]:
def predictRating(user, item):
    ratings = []
    similarities = []
    for d in reviewsPerUser[user]:
        i2 = d['product_id']
        if i2 == item:
            continue
        ratings.append(d['star_rating']) #rating of user for item i2
        similarities.append(Jaccard(usersPerItem[item], usersPerItem[i2])) #similarity of item i2 to item for which we want to predict
    if (sum(similarities) > 0):
        weightedRatings = [(x*y) for x, y in zip(ratings, similarities)] #weighted ratings
        return sum(weightedRatings) / sum(similarities) #weighted average 
    else:
        # User hasn't rated any similar items
        return ratingMean

In [121]:
dataset[1]


{'marketplace': 'US',
 'customer_id': '14640079',
 'review_id': 'RZSL0BALIYUNU',
 'product_id': 'B003LRN53I',
 'product_parent': '986692292',
 'product_title': 'Sennheiser HD203 Closed-Back DJ Headphones',
 'product_category': 'Musical Instruments',
 'star_rating': 5,
 'helpful_votes': 0,
 'total_votes': 0,
 'vine': 'N',
 'verified_purchase': 'Y',
 'review_headline': 'Five Stars',
 'review_body': 'Nice headphones at a reasonable price.',
 'review_date': '2015-08-31'}

Predict a rating for a particular user/item pair

In [122]:
u, i = dataset[1]['customer_id'], dataset[1]['product_id']


In [123]:
predictRating(u, i)


5.0

### Evaluate across the entire corpus

#### Compute the MSE for a model based on this heuristic

In [124]:
def MSE(predictions, labels):
    differences = [(x-y)**2 for x, y in zip(predictions, labels)]
    return sum(differences) / len(differences)


Compared to a trivial predictor which always predicts the mean

In [125]:
alwaysPredictMean = [ratingMean for d in dataset]


Get predictions for all instances (fairly slow!)

In [126]:
simPredictions = [predictRating(
    d['customer_id'], d['product_id']) for d in dataset]


In [127]:
labels = [d['star_rating'] for d in dataset]


In [128]:
MSE(alwaysPredictMean, labels)


1.4796142779712909

In [129]:
MSE(simPredictions, labels)

1.6146130004373205

#### Exercise 1: Update the code of predictRating(user, item) to implement a prediction based on normalized ratings 
(PML - Equation 4.22)

Hint: use the average rating of each user over all items using the itemAverages\[ \] list for each item (i2)

In [130]:
# updated rating
def predictRating(user, item):
    ratings = []
    similarities = []
    for d in reviewsPerUser[user]:
        i2 = d['product_id']
        if i2 == item:
            continue
        ratings.append(d['star_rating'] - itemAverages[i2]) # subtracted mean item average 
        similarities.append(Jaccard(usersPerItem[item], usersPerItem[i2]))
    if (sum(similarities) > 0):
        weightedRatings = [(x*y) for x, y in zip(ratings, similarities)] 
        return itemAverages[item] + (sum(weightedRatings) / sum(similarities)) # added average
    else:
        return ratingMean

In [131]:
simPredictions = [predictRating(
    d['customer_id'], d['product_id']) for d in dataset]
MSE(simPredictions, labels)

1.446725779491265

We got a lower MSE!!

#### Exercise 2: Update the code of predictRating(user, item) to implement a user-based prediction (i.e. based on user-user similarities). 

(PML - Equation 4.21)

Hint: Instead of forming an item neighborhood for given item, form a user neighborhood of users who have rated this item. Iteration should now happen over the reviewsPerItem\[ \] list, and Jaccard similarities need to be calculated between users. 


In [132]:
def predictRating(user, item):
    ratings = []
    similarities = []
    for d in reviewsPerItem[item]:
        u2 = d['customer_id'] # changed to track second user
        if u2 == user:
            continue
        ratings.append(d['star_rating'] - userAverages[u2])
        similarities.append(Jaccard(itemsPerUser[user], itemsPerUser[u2])) # compares item similarity 
    if (sum(similarities) > 0):
        weightedRatings = [(x*y) for x, y in zip(ratings, similarities)]
        return userAverages[user] + (sum(weightedRatings) / sum(similarities))
    else:
        return ratingMean

In [133]:
simPredictions = [predictRating(
    d['customer_id'], d['product_id']) for d in dataset]
MSE(simPredictions, labels)

0.43586560527133666

Significantly less MSE!!

#### Question: Would the modification above work for metrics other than Jaccard? Why/Why not? 

If not, then which parts of the code need to be udpated as well?

Yes the modifications would still work for any similarity function which could be swapped out as similarity functions like cosine, pearson, and jaccard just return a probability between 0 and 1. The only part of the code that would need to be updated would be the function call when appending the similarities.